In [ ]:
import torch
from transformers import  AutoProcessor,AutoModelForMultimodalLM

MODEL_ID= 'Qwen/Qwen3.5-4B'

# 1.加载processer和模型
processer=AutoProcessor.from_pretrained(MODEL_ID)
model=AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

# 2.转为测试模式
model.eval()

### 测试1

1. 尝试中文+非思考模式。模型能够正确回答。

In [7]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试1.png'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)


Loading weights: 100%|██████████| 723/723 [00:02<00:00, 262.12it/s]


Qwen3.5 回答：
根据图片内容，我找到的是第 2 行、第 2 列的那本书：

1. 这本书的书脊是**白色**的。
2. 书脊上有可辨认的文字，准确抄写书名：**Music**。
3. 这本书位于书架的第 2 行（从上往下数）、第 2 列（从左往右数），紧挨着“Space”书的右侧和“Animals”书的左侧。



2. 尝试中文+非思考模式+直接回答指令。模型能正确回答。

In [14]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试1.png'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
请直接输出答案，而不要输出你的分析过程。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)


Qwen3.5 回答：
1. 这本书的书脊主要颜色是白色。
2. 书名是：Music
3. 这本书位于第 2 行、第 2 列。



### 测试5

1. 中文+非思考模式。翻车了。

In [9]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第15本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)

Qwen3.5 回答：
根据图片内容，从左到右依次排列，第15本书是：

1. 这本书的书脊主要颜色是**白色**（上半部分），底部有**棕色**（或深褐色）的装饰区域。

2. 书脊上的可辨认文字是：**苏东坡新传**（上），作者为李一冰著，全新增订版，四川人民出版社出版。

3. 这本书位于**第一行**（从上往下数，书架上只有一层可见的书籍），从左往右数的**第15本**位置。它紧挨着《电工电路基本原理66课》的右侧，左侧是《苏东坡新传》（下）的右侧，右侧是《新英汉小词典》。

---

✅ 最终答案：

1. 书脊主要颜色：**白色**（上半部分），底部为**棕色**。
2. 书名：**苏东坡新传**（上）
3. 位置：**第一行，从左数第15本**



2. 下面尝试中文+思考模式。未翻车。

In [11]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第15本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=True,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)

Qwen3.5 回答：
用户要求我找到书架上从左到右的第15本书，并回答关于这本书颜色、书名和位置的问题。

1.  **确定书籍总数和排列顺序：**
    *   我需要从左到右数书脊。
    *   第1本：最左边黄色的书《苏东坡传》。
    *   第2本：浅蓝色书脊《数学II（女孩）》。
    *   第3本：浅粉色书脊《数学II（女孩）》。
    *   第4本：浅紫色书脊《数学II（女孩）》。
    *   第5本：浅绿色书脊《数学II（女孩）》。
    *   第6本：红色书脊《现代汉语词典》。
    *   第7本：绿色书脊《图解电的基础知识入门》。
    *   第8本：绿色书脊《图解电机基础知识入门》。
    *   第9本：绿色书脊《图解机械基础知识入门》。
    *   第10本：白色书脊《电磁场基本原理66课》。
    *   第11本：白色书脊《电工电路基本原理66课》。
    *   第12本：白色书脊《电子电路基本原理66课》。
    *   第13本：白色书脊《苏东坡新传》（上）。
    *   第14本：白色书脊《苏东坡新传》（下）。
    *   第15本：深蓝色/黑色书脊《新英汉小词典》。

    让我再仔细数一遍，确保没有数错。
    1. 黄色《苏东坡传》
    2. 蓝《数学II》
    3. 粉《数学II》
    4. 紫《数学II》
    5. 绿《数学II》
    6. 红《现代汉语词典》
    7. 绿《图解电...》
    8. 绿《图解电机...》
    9. 绿《图解机械...》
    10. 白《电磁场...》
    11. 白《电工电路...》
    12. 白《电子电路...》
    13. 白《苏东坡新传》（上）
    14. 白《苏东坡新传》（下）
    15. 深蓝/黑《新英汉小词典》

    好的，第15本书确实是《新英汉小词典》。

2.  **分析第15本书的特征：**
    *   **书脊颜色：** 上半部分是深蓝色（或者说是黑色带点蓝），下半部分是红色/粉色，最底部有黄色/绿色条纹。整体看起来主要是深蓝色/黑色。
    *   **书名：** 书脊上有明显的白色文字“新英汉小词典”。上方还有英文“A NEW POC

3. 尝试英文+非思考模式。没有翻车，但是有一定的列举分析。

In [12]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 15th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)

Qwen3.5 回答：
Based on the image provided, here is the analysis of the bookshelf:

**1. Counting the books:**
The books are arranged in a single row on the shelf. Counting from left to right:
1.  Yellow book (Su Dongpo)
2.  Light blue book (Math II)
3.  Pinkish book (Math II)
4.  Light purple book (Math II)
5.  Greenish book (Math II)
6.  Red book (Modern Chinese Dictionary)
7.  Green book (Electrical Engineering)
8.  Green book (Motor Basics)
9.  Green book (Mechanical Basics)
10. White book (Electromagnetic Field)
11. White book (Electrical Circuit)
12. White book (Electronic Circuit)
13. White book (Su Dongpo New Biography - Vol 1)
14. White book (Su Dongpo New Biography - Vol 2)
15. **Dark Blue/Black book (New Pocket English-Chinese Dictionary)**

**2. Analysis of the 15th book:**
*   **Position:** This book is located in the **1st row** (the only visible row) and is the **15th book** from the left.
*   **Main Color:** The main color of the spine is **dark blue** (or black with blue 

4. 下面尝试英文+非思考模式+直接输入指令。翻大车了。

In [13]:
# 3.准备图片和问题
image='/home/20261024_1/Workspace/qyh/VLM-test/datasets/几何简单测试/几何测试5.jpg'
question=f'''
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 15th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
Please output the answer directly, and do not output your analysis process.
'''
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": image,
            },
            {
                "type": "text",
                "text": question,
            },
        ],
    }
]

# 4.使用 Qwen 官方 Chat Template 构造输入
inputs=processer.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
# print(f'应用了模板的input:{inputs}')
# input_str=processer.decode(inputs['input_ids'])
# print(input_str)

# 5.模型推理和解析答案
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        temperature=1.0,
        top_p=0.95,
        top_k=20,
    )

# 只解码新生成的 Token
input_length = inputs["input_ids"].shape[-1]
answer = processer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True,
)
print("Qwen3.5 回答：")
print(answer)

Qwen3.5 回答：
The 15th book is the one titled "苏东坡新传 上".
Its main spine color is white.
It is located in the second row, the 8th column from the left.

